# VisClick — Part C (step 3): EDA on each dataset

**Prerequisite:** datasets from `01_pull_and_data.ipynb` and `02_rico_zenodo_vins.ipynb` are already on Drive under `MyDrive/visclick/data/` (CLAY, RICO, UI-Vision, unified train/val/test, optional VINS).

**This notebook:** mount Drive → `git pull` → produce **EDA figures** described in `VisClick_Detailed_Plan.md` §C.4:
1. Sample size (images + instances per class)  
2. Class balance bar chart  
3. Image resolution distribution (histogram)  
4. 9 random samples with bounding boxes drawn  

All figures are saved to **`<DRIVE>/reports/figures/eda/`** so they survive runtime restarts.

**Idempotent:** every section first checks for the target `.png` files. If they already exist, that section is **skipped** unless `FORCE = True`.

**Report:** every step prints `REPORT ...` lines for `VisClick_Report_Data_Form.md` §3.2.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import os, sys, subprocess
for pkg in ("opencv-python", "matplotlib", "pandas", "pillow"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import random, glob, json, math
random.seed(0); np.random.seed(0)

DRIVE = "/content/drive/MyDrive/visclick"
DATA  = os.path.join(DRIVE, "data")
FIG   = os.path.join(DRIVE, "reports", "figures", "eda")
os.makedirs(FIG, exist_ok=True)
FORCE = False

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]

def _have(*paths):
    return all(os.path.isfile(p) for p in paths)

def _walk_files(root, exts):
    out = []
    if not os.path.isdir(root):
        return out
    for r, _d, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(exts):
                out.append(os.path.join(r, f))
    return out

def _save(fig, path):
    fig.tight_layout(); fig.savefig(path, dpi=120, bbox_inches="tight"); plt.close(fig)
    print("saved ->", path)

print("DRIVE =", DRIVE)
print("FIG   =", FIG)
print("FORCE =", FORCE)

## 3.1 — Inventory: confirm what is on Drive

Just counts files / sizes; never downloads. Prints `REPORT inventory ...` lines.


In [ ]:
def _dir_stats(p):
    n, total = 0, 0
    if os.path.isdir(p):
        for r, _d, fs in os.walk(p):
            for f in fs:
                fp = os.path.join(r, f)
                try:
                    total += os.path.getsize(fp); n += 1
                except OSError:
                    pass
    return n, total

TARGETS = {
    "clay":      os.path.join(DATA, "clay"),
    "rico":      os.path.join(DATA, "rico"),
    "rico_combined": os.path.join(DATA, "rico", "combined"),
    "ui_vision": os.path.join(DATA, "ui_vision"),
    "unified":   os.path.join(DATA, "unified"),
    "unified_train": os.path.join(DATA, "unified", "train", "images"),
    "unified_val":   os.path.join(DATA, "unified", "val",   "images"),
    "unified_test":  os.path.join(DATA, "unified", "test",  "images"),
    "vins":      os.path.join(DATA, "vins"),
}

rows = []
for k, p in TARGETS.items():
    n, b = _dir_stats(p)
    rows.append({"dataset": k, "path": p, "files": n, "size_mb": round(b/1e6, 1)})
    print(f"REPORT inventory | {k:14s} | files={n:>7d} | size_mb={b/1e6:>9.1f} | {p}")
pd.DataFrame(rows)

## 3.2 — Unified RICO+WebUI YOLO bundle (main training source)

12 source classes (per Zenodo `dataset.yaml`), mapped down to **6 VisClick classes** for figures only. Per split (`train`/`val`/`test`) we save:  
- `eda_unified_<split>_classes.png` (instance count per VisClick class)  
- `eda_unified_<split>_resolutions.png` (image W×H histogram)  
- `eda_unified_<split>_samples.png` (9 random images with bounding boxes)

Each block **skips** if its 3 PNGs already exist (and `FORCE=False`).


In [ ]:
UNIFIED_NAMES = ["Button", "Text", "Image", "Icon", "Input", "Link",
                  "Checkbox", "Toggle", "Toolbar", "Navigation", "Modal", "Tab"]
UNIFIED_TO_VIS = {
    "Button": "button", "Toggle": "button", "Tab": "button",
    "Text": "text", "Link": "text",
    "Input": "text_input",
    "Image": "icon", "Icon": "icon",
    "Toolbar": "menu", "Navigation": "menu", "Modal": "menu",
    "Checkbox": "checkbox",
}

def _yolo_label_dir(split_root):
    for cand in ("labels", os.path.join("..", "labels")):
        p = os.path.normpath(os.path.join(split_root, cand))
        if os.path.isdir(p):
            return p
    return None

def _eda_yolo_split(split_root, split_name, class_names, name_map, tag, color=(0,255,0)):
    out_cls = os.path.join(FIG, f"eda_{tag}_{split_name}_classes.png")
    out_res = os.path.join(FIG, f"eda_{tag}_{split_name}_resolutions.png")
    out_sam = os.path.join(FIG, f"eda_{tag}_{split_name}_samples.png")

    if not FORCE and _have(out_cls, out_res, out_sam):
        print(f"skip {tag}/{split_name} (figures already exist)")
        print(f"REPORT step = EDA_{tag.upper()} | split = {split_name} | status = SKIP_EXISTS")
        return

    img_dir = os.path.join(split_root, "images") if os.path.isdir(os.path.join(split_root, "images")) else split_root
    lbl_dir = _yolo_label_dir(img_dir if img_dir != split_root else split_root)
    imgs = _walk_files(img_dir, (".jpg", ".jpeg", ".png"))
    if not imgs:
        print(f"REPORT step = EDA_{tag.upper()} | split = {split_name} | status = NO_IMAGES | path = {img_dir}")
        return

    counts = {v: 0 for v in CLASSES}
    if lbl_dir is not None:
        for lp in _walk_files(lbl_dir, (".txt",)):
            try:
                with open(lp) as fh:
                    for line in fh:
                        parts = line.strip().split()
                        if not parts: continue
                        try:
                            cid = int(parts[0])
                        except ValueError:
                            continue
                        if 0 <= cid < len(class_names):
                            vis = name_map.get(class_names[cid])
                            if vis:
                                counts[vis] += 1
            except OSError:
                pass
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(list(counts.keys()), list(counts.values()), color="#3a7bd5")
    ax.set_title(f"{tag} / {split_name} — instances per VisClick class")
    ax.set_ylabel("# instances"); ax.tick_params(axis='x', rotation=20)
    _save(fig, out_cls)

    sample_for_res = imgs if len(imgs) <= 4000 else random.sample(imgs, 4000)
    ws, hs = [], []
    for ip in sample_for_res:
        try:
            with Image.open(ip) as im:
                w, h = im.size
            ws.append(w); hs.append(h)
        except Exception:
            pass
    fig, axs = plt.subplots(1, 2, figsize=(10, 3))
    axs[0].hist(ws, bins=40, color="#3a7bd5"); axs[0].set_title("width")
    axs[1].hist(hs, bins=40, color="#3a7bd5"); axs[1].set_title("height")
    fig.suptitle(f"{tag} / {split_name} — image resolution (sample {len(ws)} of {len(imgs)})")
    _save(fig, out_res)

    pick = random.sample(imgs, k=min(9, len(imgs)))
    fig, axs = plt.subplots(3, 3, figsize=(9, 9))
    for ax, ip in zip(axs.flat, pick):
        im = cv2.imread(ip)
        if im is None:
            ax.axis("off"); continue
        H, W = im.shape[:2]
        if lbl_dir is not None:
            stem = os.path.splitext(os.path.basename(ip))[0]
            lp = os.path.join(lbl_dir, stem + ".txt")
            if os.path.isfile(lp):
                with open(lp) as fh:
                    for line in fh:
                        p = line.strip().split()
                        if len(p) < 5: continue
                        try:
                            _, x, y, bw, bh = int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])
                        except ValueError:
                            continue
                        x1, y1 = int((x - bw/2)*W), int((y - bh/2)*H)
                        x2, y2 = int((x + bw/2)*W), int((y + bh/2)*H)
                        cv2.rectangle(im, (x1, y1), (x2, y2), color, 2)
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
    fig.suptitle(f"{tag} / {split_name} — 9 random samples with boxes")
    _save(fig, out_sam)

    total_inst = sum(counts.values())
    print(f"REPORT step = EDA_{tag.upper()} | split = {split_name} | images = {len(imgs)} | instances = {total_inst}")
    for k, v in counts.items():
        print(f"REPORT step = EDA_{tag.upper()} | split = {split_name} | class = {k:11s} | n = {v}")

for sp in ("train", "val", "test"):
    sr = os.path.join(DATA, "unified", sp)
    if os.path.isdir(sr):
        _eda_yolo_split(sr, sp, UNIFIED_NAMES, UNIFIED_TO_VIS, tag="unified")
    else:
        print(f"REPORT step = EDA_UNIFIED | split = {sp} | status = MISSING | path = {sr}")

## 3.3 — CLAY (denoised RICO labels)

CLAY ships **labels only** as `clay_labels.csv` (or `clay_labels_<split>.csv`); images come from RICO. Figures: VisClick-class balance from `clay_labels*.csv` (skip if `eda_clay_classes.png` exists).


In [ ]:
CLAY_TO_VIS = {
    "BUTTON": "button", "RADIO_BUTTON_GROUP": "button", "TOGGLE_SWITCH": "button",
    "TEXT": "text", "LABEL": "text",
    "INPUT": "text_input", "EDIT_TEXT": "text_input",
    "ICON": "icon", "IMAGE_BUTTON": "icon",
    "LIST_ITEM": "menu", "MENU_ITEM": "menu", "NAV_DRAWER": "menu",
    "CHECKBOX": "checkbox", "CHECKED_TEXTVIEW": "checkbox",
}

out_cls = os.path.join(FIG, "eda_clay_classes.png")
if not FORCE and os.path.isfile(out_cls):
    print("skip CLAY (figure already exists)")
    print("REPORT step = EDA_CLAY | status = SKIP_EXISTS")
else:
    csvs = _walk_files(os.path.join(DATA, "clay"), (".csv",))
    label_csvs = [p for p in csvs if "label" in os.path.basename(p).lower()]
    if not label_csvs:
        print("REPORT step = EDA_CLAY | status = NO_CSV | path =", os.path.join(DATA, "clay"))
    else:
        counts = {v: 0 for v in CLASSES}
        total_rows = 0
        for cp in label_csvs:
            try:
                df = pd.read_csv(cp, low_memory=False)
            except Exception as e:
                print("skip", cp, ":", e); continue
            label_col = next((c for c in df.columns if c.lower() in ("label", "class", "category", "node_class")), None)
            if label_col is None:
                print("no label column in", cp, "— columns:", list(df.columns)[:6]); continue
            total_rows += len(df)
            for v in df[label_col].astype(str).str.upper().value_counts().items():
                src, n = v
                vis = CLAY_TO_VIS.get(src)
                if vis:
                    counts[vis] += int(n)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(list(counts.keys()), list(counts.values()), color="#d55e00")
        ax.set_title("CLAY — instances per VisClick class (mapped from raw labels)")
        ax.set_ylabel("# instances"); ax.tick_params(axis='x', rotation=20)
        _save(fig, out_cls)
        print(f"REPORT step = EDA_CLAY | csvs = {len(label_csvs)} | rows = {total_rows} | mapped = {sum(counts.values())}")
        for k, v in counts.items():
            print(f"REPORT step = EDA_CLAY | class = {k:11s} | n = {v}")

## 3.4 — RICO (UI screenshots, no class labels here)

RICO `combined/` holds `.jpg` screenshots + view-hierarchy `.json`. We only chart **resolutions** + 9 random samples (skip if `eda_rico_resolutions.png` and `eda_rico_samples.png` exist).


In [ ]:
out_res = os.path.join(FIG, "eda_rico_resolutions.png")
out_sam = os.path.join(FIG, "eda_rico_samples.png")
if not FORCE and _have(out_res, out_sam):
    print("skip RICO (figures already exist)")
    print("REPORT step = EDA_RICO | status = SKIP_EXISTS")
else:
    rico_combined = os.path.join(DATA, "rico", "combined")
    imgs = _walk_files(rico_combined, (".jpg", ".jpeg", ".png"))
    if not imgs:
        print("REPORT step = EDA_RICO | status = NO_IMAGES | path =", rico_combined)
    else:
        sample = imgs if len(imgs) <= 4000 else random.sample(imgs, 4000)
        ws, hs = [], []
        for ip in sample:
            try:
                with Image.open(ip) as im:
                    w, h = im.size
                ws.append(w); hs.append(h)
            except Exception:
                pass
        fig, axs = plt.subplots(1, 2, figsize=(10, 3))
        axs[0].hist(ws, bins=40, color="#3a7bd5"); axs[0].set_title("width")
        axs[1].hist(hs, bins=40, color="#3a7bd5"); axs[1].set_title("height")
        fig.suptitle(f"RICO/combined — image resolution (sample {len(ws)} of {len(imgs)})")
        _save(fig, out_res)

        pick = random.sample(imgs, k=min(9, len(imgs)))
        fig, axs = plt.subplots(3, 3, figsize=(9, 9))
        for ax, ip in zip(axs.flat, pick):
            im = cv2.imread(ip)
            if im is None:
                ax.axis("off"); continue
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
        fig.suptitle("RICO/combined — 9 random samples")
        _save(fig, out_sam)
        print(f"REPORT step = EDA_RICO | images = {len(imgs)} | sample_for_resolution = {len(ws)}")

## 3.5 — UI-Vision (desktop benchmark)

External test set. Many shards under `data/ui_vision/`. Save resolution histogram + 9 random samples (skip if both PNGs exist).


In [ ]:
out_res = os.path.join(FIG, "eda_uivision_resolutions.png")
out_sam = os.path.join(FIG, "eda_uivision_samples.png")
if not FORCE and _have(out_res, out_sam):
    print("skip UI-Vision (figures already exist)")
    print("REPORT step = EDA_UIVISION | status = SKIP_EXISTS")
else:
    root = os.path.join(DATA, "ui_vision")
    imgs = _walk_files(root, (".png", ".jpg", ".jpeg"))
    if not imgs:
        print("REPORT step = EDA_UIVISION | status = NO_IMAGES | path =", root)
    else:
        sample = imgs if len(imgs) <= 4000 else random.sample(imgs, 4000)
        ws, hs = [], []
        for ip in sample:
            try:
                with Image.open(ip) as im:
                    w, h = im.size
                ws.append(w); hs.append(h)
            except Exception:
                pass
        fig, axs = plt.subplots(1, 2, figsize=(10, 3))
        axs[0].hist(ws, bins=40, color="#0a9396"); axs[0].set_title("width")
        axs[1].hist(hs, bins=40, color="#0a9396"); axs[1].set_title("height")
        fig.suptitle(f"UI-Vision — image resolution (sample {len(ws)} of {len(imgs)})")
        _save(fig, out_res)

        pick = random.sample(imgs, k=min(9, len(imgs)))
        fig, axs = plt.subplots(3, 3, figsize=(9, 9))
        for ax, ip in zip(axs.flat, pick):
            im = cv2.imread(ip)
            if im is None:
                ax.axis("off"); continue
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
        fig.suptitle("UI-Vision — 9 random samples")
        _save(fig, out_sam)
        print(f"REPORT step = EDA_UIVISION | images = {len(imgs)} | sample_for_resolution = {len(ws)}")

## 3.6 — VINS (only if images are present)

VINS images come from an external archive; if `data/vins/` is empty this block is skipped automatically.


In [ ]:
out_res = os.path.join(FIG, "eda_vins_resolutions.png")
out_sam = os.path.join(FIG, "eda_vins_samples.png")
if not FORCE and _have(out_res, out_sam):
    print("skip VINS (figures already exist)")
    print("REPORT step = EDA_VINS | status = SKIP_EXISTS")
else:
    root = os.path.join(DATA, "vins")
    imgs = _walk_files(root, (".png", ".jpg", ".jpeg"))
    if not imgs:
        print("REPORT step = EDA_VINS | status = NO_IMAGES | path =", root)
    else:
        sample = imgs if len(imgs) <= 4000 else random.sample(imgs, 4000)
        ws, hs = [], []
        for ip in sample:
            try:
                with Image.open(ip) as im:
                    w, h = im.size
                ws.append(w); hs.append(h)
            except Exception:
                pass
        fig, axs = plt.subplots(1, 2, figsize=(10, 3))
        axs[0].hist(ws, bins=40, color="#9d4edd"); axs[0].set_title("width")
        axs[1].hist(hs, bins=40, color="#9d4edd"); axs[1].set_title("height")
        fig.suptitle(f"VINS — image resolution (sample {len(ws)} of {len(imgs)})")
        _save(fig, out_res)

        pick = random.sample(imgs, k=min(9, len(imgs)))
        fig, axs = plt.subplots(3, 3, figsize=(9, 9))
        for ax, ip in zip(axs.flat, pick):
            im = cv2.imread(ip)
            if im is None:
                ax.axis("off"); continue
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.axis("off")
        fig.suptitle("VINS — 9 random samples")
        _save(fig, out_sam)
        print(f"REPORT step = EDA_VINS | images = {len(imgs)} | sample_for_resolution = {len(ws)}")

## 3.7 — Final REPORT

Lists every figure produced under `<DRIVE>/reports/figures/eda/`. Copy the `REPORT figure ...` lines into `VisClick_Report_Data_Form.md` §3.2.


In [ ]:
pngs = sorted(_walk_files(FIG, (".png",)))
for p in pngs:
    print("REPORT figure |", os.path.basename(p), "|", p, "| size_kb =", round(os.path.getsize(p)/1024, 1))
print("REPORT step = EDA_DONE | n_figures =", len(pngs), "| dir =", FIG)